# E16b — BackboneConceptLM generation playground

Interactive continuation from the **E16b best checkpoint** (Gemma-3-1B + shared depth-recurrent concepts, 4K / Muon / long-doc mix).

| | |
|---|---|
| Run | `backbone_concept_gemma_3_1b_pt_K512_concept_20260718_150850` |
| Checkpoint | `checkpoint-7900` (best by eval_loss) |
| Architecture | frozen Gemma-3-1B + LoRA r16, C=128, K=512, `shared_depth_recurrent` |
| W&B | [link](https://wandb.ai/ksopyla/MrCogito/runs/backbone_concept_gemma_3_1b_pt_K512_concept_20260718_150850) |

## How generation works here

This is a **block-recurrent causal LM**, not the E05 encode→decode concept bottleneck:

1. Your prompt tokens are consumed in **512-token blocks**.
2. After each global Gemma layer, concepts are **read** (gated) and **written** (BiXT).
3. The next token is predicted from the last position's hidden state via the Gemma LM head.

So this is true **continuation** (the model sees your exact prompt tokens), with a compact recurrent concept state carrying information across blocks.

Consequences:
- Base LM, **not instruction-tuned** — use continuation prompts (`Once upon a time…`), not chat instructions.
- **No KV cache**: each new token re-runs the growing prefix. Keep `max_new_tokens` ≤ ~128 on MPS for snappy play.
- Long prompts (>1–2K tokens) will be slower; the interesting concept-memory regime is multi-block (≥1024).

In [ ]:
import os, sys, time
from pathlib import Path

REPO = Path('/Users/ksirg/devel/MrCogito')
if not (REPO / 'nn').is_dir():
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))

import torch
from transformers import AutoTokenizer

from nn.backbone_concept_lm import BackboneConceptLM

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

CKPT = REPO / 'Cache/Training/backbone_concept_gemma_3_1b_pt_K512_concept_20260718_150850/checkpoint-7900'
assert CKPT.is_dir(), f'Missing checkpoint at {CKPT} — rsync from Odra first'
assert (CKPT / 'model.safetensors').is_file()

print(f'repo:   {REPO}')
print(f'device: {DEVICE} | torch {torch.__version__}')
print(f'ckpt:   {CKPT}')

## Load model + tokenizer

On Apple Silicon, prefer `float16` on MPS (bf16 support varies). First load takes a bit while weights map onto the device.

In [ ]:
DTYPE = torch.float16 if DEVICE in {'mps', 'cuda'} else torch.float32

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained('google/gemma-3-1b-pt')
model = BackboneConceptLM.from_pretrained(CKPT, dtype=DTYPE)
model = model.to(DEVICE).eval()
print(f'loaded in {time.time() - t0:.1f}s | dtype={DTYPE} | params={sum(p.numel() for p in model.parameters()):,}')
print('concept_io_mode:', model.config.concept_io_mode)
print('concept_num / block:', model.config.concept_num, model.config.concept_block)
print('gates:', {k: round(v, 4) for k, v in model.concept_gate_metrics().items() if 'layer' not in k})

## Generation helpers

In [ ]:
def continue_text(
    prompt: str,
    *,
    max_new_tokens: int = 64,
    do_sample: bool = True,
    temperature: float = 0.8,
    top_k: int = 50,
    top_p: float = 0.95,
    seed: int | None = 0,
    max_prompt_tokens: int = 1024,
):
    """Continue ``prompt`` with the block-recurrent concept LM."""
    if seed is not None:
        torch.manual_seed(seed)
    enc = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=max_prompt_tokens,
        add_special_tokens=True,
    )
    input_ids = enc['input_ids'].to(DEVICE)
    attention_mask = enc['attention_mask'].to(DEVICE)
    t0 = time.time()
    out = model.generate(
        input_ids,
        attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )
    dt = time.time() - t0
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    new_ids = out[0, input_ids.shape[1]:]
    continuation = tokenizer.decode(new_ids, skip_special_tokens=True)
    print(f'[{dt:.1f}s | prompt {input_ids.shape[1]} tok → +{len(new_ids)} tok | '
          f'{"sample" if do_sample else "greedy"}]')
    return {'full': full, 'continuation': continuation, 'n_new': int(len(new_ids))}


def show(prompt, **kw):
    r = continue_text(prompt, **kw)
    print('--- prompt ---')
    print(prompt)
    print('--- continuation ---')
    print(r['continuation'])
    return r

## Short continuations (local fluency)

These stay inside one or two blocks — mostly tests local Gemma+LoRA fluency, not long-range concept memory.

In [ ]:
show(
    'The theory of relativity states that',
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    seed=0,
)

In [ ]:
show(
    'Once upon a time in a small village by the sea,',
    max_new_tokens=80,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    seed=1,
)

In [ ]:
# Greedy (more deterministic / sometimes more repetitive)
show(
    'In machine learning, a concept bottleneck is',
    max_new_tokens=50,
    do_sample=False,
)

## Multi-block prompt (concept-memory regime)

Paste a longer passage (≥ ~600–1200 tokens) so the model must cross at least one block boundary. Ask it to continue *after* details that only appeared early in the prompt — that is closer to what the ablation metrics measure.

Tip: start from a Wikipedia / book paragraph, or duplicate a short fact far before the end.

In [ ]:
# Early fact buried before filler, then ask for continuation that needs the fact.
FACT = (
    'The secret passphrase for the lighthouse keepers was "amber gull". '
    'Only the night watch knew it.\n\n'
)
FILLER = (
    'The coastline stretched for miles under a grey sky. Waves broke against '
    'the rocks while gulls circled the cliffs. Fishermen hauled nets onto '
    'the pier and talked about the weather, the tides, and the catch. '
) * 40  # inflate to force multiple 512-token blocks
TAIL = (
    '\n\nWhen the new apprentice asked for the passphrase, the keeper replied:'
)
long_prompt = FACT + FILLER + TAIL
print(f'approx chars={len(long_prompt):,} | '
      f'approx tokens≈{len(tokenizer(long_prompt)["input_ids"]):,}')

show(
    long_prompt,
    max_new_tokens=40,
    max_prompt_tokens=2048,
    do_sample=True,
    temperature=0.7,
    seed=2,
)

## Your turn

Edit the prompt below and re-run. Try:
- factual completion vs story continuation
- greedy vs sampling (`do_sample=False` / `True`)
- burying a detail early and probing it late (multi-block)

In [ ]:
show(
    '''Replace this with your own prompt.''',
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    seed=42,
)

## Notes / expectations

- E16b cleared **concept causal-use** ablations (Δshuffle_beyond ≈ 2.5 nats offline) — that measures teacher-forced CE under concept corruption, **not** free-running literary quality.
- Free generation can still be repetitive or bland; judge fluency separately from the mechanism gates.
- For a cleaner long-range probe, prefer the delayed-recall / ablation scripts over vibes from short samples.
- Related: `playground/e05_generation_comparison.ipynb` (different architecture: encode→decode concepts).